In [1]:
from dask.distributed import Client

client = Client(threads_per_worker=1)
client

2026-03-12 13:39:27,884 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/tornado/web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expir

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 11
Total threads: 11,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:62331,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:62372,Total threads: 1
Dashboard: http://127.0.0.1:62373/status,Memory: 1.64 GiB
Nanny: tcp://127.0.0.1:62334,


2026-03-12 13:39:31,899 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/tornado/web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
  File "/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expir

In [5]:
import warnings
import os
import xarray as xr
from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry

import numcodecs.zarr3  # Register the zarr3 codec

from dotenv import load_dotenv

load_dotenv()

access_key_id = os.environ.get("ACCESS_KEY_ID")
secret_access_key = os.environ.get("SECRET_ACCESS_KEY")
endpoint = "https://projects.pawsey.org.au"
bucket = "s3://1deg"

# create s3 store with aws-style credentials
store = S3Store.from_url(
    f"{bucket}",
    endpoint=endpoint,
    access_key_id=access_key_id,
    secret_access_key=secret_access_key,
)

registry = ObjectStoreRegistry({f"{bucket}": store})

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)
parser = HDFParser()

We now have proof that we can open that netcdf with virtualizarr. Let's now try opening the entire contents of that bucket as a single virtual dataset.

Following that, we'll try to save it via icechunk

In [3]:
import obstore

flist = []
for batch in obstore.list(store):
    for obj in batch:
        if not obj.get("path","icechunk").startswith("icechunk"):
            flist.append(obj.get("path"))

print(flist)

['output1830/ocean_year.nc', 'output1831/ocean_year.nc', 'output1832/ocean_year.nc', 'output1833/ocean_year.nc', 'output1834/ocean_year.nc', 'output1835/ocean_year.nc', 'output1836/ocean_year.nc', 'output1837/ocean_year.nc', 'output1838/ocean_year.nc', 'output1839/ocean_year.nc', 'output1840/ocean_year.nc', 'output1841/ocean_year.nc', 'output1842/ocean_year.nc', 'output1843/ocean_year.nc', 'output1844/ocean_year.nc', 'output1845/ocean_year.nc', 'output1846/ocean_year.nc', 'output1847/ocean_year.nc', 'output1848/ocean_year.nc', 'output1849/ocean_year.nc', 'output1850/ocean_year.nc', 'output1851/ocean_year.nc', 'output1852/ocean_year.nc', 'output1853/ocean_year.nc', 'output1854/ocean_year.nc', 'output1855/ocean_year.nc', 'output1856/ocean_year.nc', 'output1857/ocean_year.nc', 'output1858/ocean_year.nc', 'output1859/ocean_year.nc', 'output1860/ocean_year.nc', 'output1861/ocean_year.nc', 'output1862/ocean_year.nc', 'output1863/ocean_year.nc', 'output1864/ocean_year.nc', 'output1865/ocean_y

In [4]:
vds = open_virtual_dataset(
    url=f"{bucket}/{flist[0]}",
    parser=parser,
    registry=registry,
    drop_variables=["salt","temp","age_global"]
)

/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/virtualizarr/xarray.py:343: FutureWarning: In a future version, xarray will not decode the variable 'average_DT' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  with xr.open_zarr(
/Users/u1166368/PawserVirtualisationTests/.pixi/envs/default/lib/python3.14/site-packages/virtualizarr/xarray.py:343: FutureWarning: In a future version, xarray will not decode the variable 'time_bounds' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by defaul

In [7]:
vds

<xarray.Dataset> Size: 2MB
Dimensions:         (xt_ocean: 360, yt_ocean: 300, st_ocean: 50,
                     st_edges_ocean: 51, time: 1, nv: 2)
Coordinates:
  * xt_ocean        (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 78.5 79.5
  * yt_ocean        (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 89.32 89.77
  * st_ocean        (st_ocean) float64 400B 1.152 3.649 ... 5.034e+03 5.254e+03
  * st_edges_ocean  (st_edges_ocean) float64 408B 0.0 2.303 ... 5.363e+03
  * time            (time) datetime64[ns] 8B 1958-06-30T12:00:00
  * nv              (nv) float64 16B 1.0 2.0
Data variables:
    sst             (time, yt_ocean, xt_ocean) float32 432kB ManifestArray<sh...
    sss             (time, yt_ocean, xt_ocean) float32 432kB ManifestArray<sh...
    mld             (time, yt_ocean, xt_ocean) float32 432kB ManifestArray<sh...
    sea_level       (time, yt_ocean, xt_ocean) float32 432kB ManifestArray<sh...
    average_T1      (time) float64 8B ManifestArray<shape=(1,), dtype=float64...
    average_T2      (time) float64 8B ManifestArray<shape=(1,), dtype=float64...
    average_DT      (time) float64 8B ManifestArray<shape=(1,), dtype=float64...
    time_bounds     (time, nv) float64 16B ManifestArray<shape=(1, 2), dtype=...
Attributes:
    filename:   ocean_year.nc
    title:      ACCESS-OM2-BGC
    grid_type:  mosaic
    grid_tile:  1

In [ ]:
combined_vds = open_virtual_mfdataset(
    urls=[f"{bucket}/{f}" for f in flist],
    parser=parser,
    registry=registry,
    combine="nested",
    concat_dim="time",
    parallel="dask",
    compat="override",
    coords=["time",],
    data_vars=['sss','sst']
    drop_variables=["average_T1", "average_T2", "average_DT"],  # add all offenders
)

In [14]:
combined_vds

<xarray.Dataset> Size: 4GB
Dimensions:         (time: 61, st_ocean: 50, yt_ocean: 300, xt_ocean: 360,
                     nv: 2, st_edges_ocean: 51)
Coordinates:
  * xt_ocean        (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 78.5 79.5
  * yt_ocean        (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 89.32 89.77
  * st_ocean        (st_ocean) float64 400B 1.152 3.649 ... 5.034e+03 5.254e+03
  * st_edges_ocean  (st_edges_ocean) float64 408B 0.0 2.303 ... 5.363e+03
  * time            (time) datetime64[ns] 488B 1958-06-30T12:00:00 ... 2018-0...
  * nv              (nv) float64 16B 1.0 2.0
Data variables:
    temp            (time, st_ocean, yt_ocean, xt_ocean) float32 1GB Manifest...
    salt            (time, st_ocean, yt_ocean, xt_ocean) float32 1GB Manifest...
    sst             (time, yt_ocean, xt_ocean) float32 26MB ManifestArray<sha...
    sss             (time, yt_ocean, xt_ocean) float32 26MB ManifestArray<sha...
    age_global      (time, st_ocean, yt_ocean, xt_ocean) float32 1GB Manifest...
    mld             (time, yt_ocean, xt_ocean) float32 26MB ManifestArray<sha...
    sea_level       (time, yt_ocean, xt_ocean) float32 26MB ManifestArray<sha...
    time_bounds     (time, nv) float64 976B ManifestArray<shape=(61, 2), dtyp...
Attributes:
    filename:   ocean_year.nc
    title:      ACCESS-OM2-BGC
    grid_type:  mosaic
    grid_tile:  1

In [ ]:
# Copied from `~/scratch/virtualizarr/test_nb.ipynb`
# Just using to save the combined virtual dataset as a zarr via icechunk, so we
# don't have to fetch it from pawsey to rebuild it there if something crashes
import xarray as xr
import icechunk
from pathlib import Path

if Path("/Users/u1166368/scratch/virtualizarr/pawsey/1deg").exists():
    import shutil
    shutil.rmtree("/Users/u1166368/scratch/virtualizarr/pawsey/1deg")

# Create a new repository instance with virtual chunk container permissions for reading
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket}/",
        store=icechunk.s3_store(
            endpoint_url=endpoint,              # "https://projects.pawsey.org.au"
            s3_compatible=True,
            force_path_style=True,
        )
    ),
)

# credentials = icechunk.containers_credentials(
#     { "s3://my_s3_bucket": icechunk.s3_credentials(bucket="my-s3-bucket", region="us-east-1"),
#       "s3://my_other_s3_bucket": icechunk.s3_credentials(bucket="my-other-s3-bucket", region="us-west-2"),
#     }
# )

credentials = icechunk.containers_credentials(
    { 
        bucket: icechunk.s3_credentials(
            access_key_id=access_key_id, 
            secret_access_key=secret_access_key,
        )
    }
)

# Open the repository with the config that includes virtual chunk permissions
storage = icechunk.local_filesystem_storage("/Users/u1166368/scratch/virtualizarr/pawsey/1deg")
repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)

# Write your virtual dataset to the repository
write_session = repo.writable_session("main")
combined_vds.vz.to_icechunk(write_session.store)
write_session.commit("Write Pawsey 0deg virtual dataset to local zarr store for testing")

In [ ]:
import xarray as xr
import icechunk
from pathlib import Path

# Create a new repository instance with virtual chunk container permissions for reading
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket}/",
        store=icechunk.s3_store(
            endpoint_url=endpoint,              # "https://projects.pawsey.org.au"
            s3_compatible=True,
            force_path_style=True,
        )
    ),
)

credentials = icechunk.containers_credentials(
    { 
        bucket: icechunk.s3_credentials(
            access_key_id=access_key_id, 
            secret_access_key=secret_access_key,
        )
    }
)

# Open the repository with the config that includes virtual chunk permissions
storage = icechunk.local_filesystem_storage("/Users/u1166368/scratch/virtualizarr/pawsey/01deg")
repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)

# Write your virtual dataset to the repository
session = repo.readonly_session("main")
xr.open_zarr(session.store, consolidated=False)['sst_m'].isel(time=0).plot()

In [ ]:
import matplotlib.pyplot as plt
_arr = xr.open_zarr(session.store, consolidated=False)['sst_m'].data.blocks[0,0,0].compute().squeeze()
plt.pcolor(_arr)

In [ ]:
# Now put the virtual dataset in the bucket alongside the netcdf files
import xarray as xr
import icechunk
from pathlib import Path

# Create a new repository instance with virtual chunk container permissions for reading
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket}/",
        store=icechunk.s3_store(
            endpoint_url=endpoint,              # "https://projects.pawsey.org.au"
            s3_compatible=True,
            force_path_style=True,
        )
    ),
)

credentials = icechunk.containers_credentials(
    { 
        bucket: icechunk.s3_credentials(
            access_key_id=access_key_id, 
            secret_access_key=secret_access_key,
        )
    }
)

# Open the repository with the config that includes virtual chunk permissions
storage = icechunk.s3_storage(
    bucket='1deg',
    prefix='icechunk',
    endpoint_url=endpoint,
    access_key_id=access_key_id,
    secret_access_key=secret_access_key,
    force_path_style=True,
)
repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)

# Write your virtual dataset to the repository
write_session = repo.writable_session("main")
combined_vds.vz.to_icechunk(write_session.store)
write_session.commit("Write Pawsey 1deg virtual dataset to pawsey icechunk store for testing")

In [ ]:
from dask.distributed import Client

client = Client(threads_per_worker=1)
client

In [ ]:
import icechunk
from dotenv import load_dotenv
load_dotenv()

access_key_id = os.environ.get("ACCESS_KEY_ID")
secret_access_key = os.environ.get("SECRET_ACCESS_KEY")

endpoint = "https://projects.pawsey.org.au"
bucket = "s3://1deg"

storage = icechunk.s3_storage(
    bucket='1deg',
    prefix='icechunk',
    endpoint_url=endpoint,
    access_key_id=access_key_id,
    secret_access_key=secret_access_key,
    force_path_style=True,
)

credentials = icechunk.containers_credentials(
    { 
        bucket: icechunk.s3_credentials(
            access_key_id=access_key_id, 
            secret_access_key=secret_access_key,
        )
    }
)

repo = icechunk.Repository.open(storage=storage, config=None, authorize_virtual_chunk_access=credentials)

session = repo.readonly_session("main")

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

_arr = xr.open_zarr(session.store, consolidated=False)['sst'].data.blocks[0,0,0].compute().squeeze()
plt.pcolor(_arr)

In [ ]:
xr.open_zarr(session.store, consolidated=False)

In [ ]:
%%timeit
da = xr.open_zarr(session.store, consolidated=False)['sst'].isel(time=0).compute()
#da.plot()